# AWS SageMaker Feature Store 1. Features for Training

Turning raw orders into customer features, keeping them in SageMaker Feature Store, and building a training set from the offline store that only uses what was known at the time of each order.

---

Setup values and reference snippets are in code cells for easy copying; they are not meant to be executed. Run the Python steps individually. Saved example outputs appear in Markdown below their code.


## **Agenda**

1. [What a feature store is](#scrollTo=c003)
2. [The project](#scrollTo=c004)
3. [Setup](#scrollTo=c005)
4. [From orders to feature rows](#scrollTo=c008)
5. [A feature group](#scrollTo=c013)
6. [Writing the rows](#scrollTo=c016)
7. [The offline store](#scrollTo=c019)
8. [A training set that only uses the past](#scrollTo=c024)
9. [Training on SageMaker](#scrollTo=c029)
10. [Tear down](#scrollTo=c034)
11. [Class summary](#scrollTo=c037)
12. [References and further reading](#scrollTo=c038)

## **Preface**

A model almost never learns from raw rows as they are. It learns from **features**, values worked out from the raw data that say something useful. "This customer has returned 11 of their 14 orders" is a feature. It is not in any single order. Someone has to work it out from all of their orders.

Working it out is the easy part. The hard part comes later, when the same feature is needed in two places. Training needs it for thousands of past orders at once. The live app needs it for one new order, in a few milliseconds. And the training rows must use each customer's history as it was on the day of each order, not as it is today. A **feature store** is a service built for exactly this. You write each feature row once, with the time it became true, and both sides read from the same place.

This class covers the training side. You will compute customer features from a year of orders, store them in SageMaker Feature Store, read them back from the **offline store**, and train a model on them. The next class covers the live side, the **online store**, and puts the model behind an endpoint.

**[Please Note:]** The Python code runs inside SageMaker. Section 3 sets up a Studio domain and a JupyterLab space, then uploads this notebook into JupyterLab. From then on, stay in JupyterLab: read the explanation, run the Python step with **Shift+Enter**, and talk through the example output beneath it. The AWS console is used only in section 3 and at the very end. The displayed outputs come from the original recorded run, so IDs, times and some timings in yours will differ. Make sure the account has a default VPC before class (section 3.1 says how).

**By the end you will be able to do all of this.**

1. Explain what a feature, a feature group, a record identifier and an event time are.
2. Turn raw orders into feature rows that record when each value became true.
3. Create a feature group and write rows into it.
4. Read the offline store back with a SQL query.
5. Build a training set that gives each order only the features known before it, and show what goes wrong without that.
6. Train a model on SageMaker from that training set.

**Prerequisites.** S3 buckets, training jobs and AUC, from the SageMaker classes. Basic pandas.

## **1. What a feature store is**

### 1.1 Features

Here is one order from an online shop.


In [ ]:
order_id    customer_id  order_date  category  price  payment_method  returned
ORD-210337  C-00001      2026-08-10  apparel   1499   cod             1

A model that guesses whether this order will be returned can use its own columns, such as the category and the price. It does much better if it also knows the customer. Does this customer return things often? Have they been shopping here for years? Those answers are not in the order. They come from all of their earlier orders, added up. That is what a feature is, a value computed from raw data, such as `orders_so_far`, `returns_so_far` and `return_rate`.

### 1.2 Two problems, one place

Teams that compute features by hand run into two problems.

The first is **two copies of the same feature**. The training code works out `return_rate` from the data warehouse. Months later the live app works it out again from its own database, written by someone else. One counts the order being placed and the other does not, or one rounds and the other does not. Nothing fails. The model just gets slightly different numbers live than it saw in training.

The second is **time**. A training row for an order placed on 10 August must hold the customer's history as it was on 10 August. If it holds their history as it is today, it includes returns that happened after the order, sometimes the return of that very order. The model learns from the future, scores brilliantly in testing and disappoints live.

A feature store answers both. Features are computed once and written to one place. Every row is stored with the time it became true. Training reads the full history from the **offline store** and can pick the right row for any date. The live app reads the newest row for one customer from the **online store** in milliseconds.

![c13_feature_store_map.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/450/original/c13_feature_store_map.png?1793273310)

This class follows the top path, from orders to a trained model. The next class follows the bottom path, from the online store to a live prediction.


## **2. The project**

The shop is the one from the SageMaker classes, which wants to know at checkout whether an order is likely to come back. Returns cost the shop shipping both ways and repacking. If it knows an order is risky, it can show a size guide, ask for prepayment instead of cash on delivery, or check the address.

What is new is the customer. The data now holds 12,000 orders from 1,440 customers, placed between October 2025 and September 2026, and each order says who placed it. About a third of all orders come back.

| Column | What it holds |
|---|---|
| `order_id`, `customer_id` | Who ordered what |
| `order_date` | The day of the order |
| `category`, `price`, `discount_pct` | What was bought and for how much |
| `delivery_days`, `payment_method`, `is_gift`, `seller_rating` | How it was sold and sent |
| `signup_date` | When the customer joined the shop |
| `returned` | The label, 1 if the order came back |

One detail matters for everything that follows. **A return is only known 14 days after the order**, when the parcel arrives back at the warehouse. On the day of the order, nobody knows yet whether it will be returned. The features must respect that.

The plan for the class follows the top path of the picture. Compute customer features from the orders, write them to a feature group, read them back from the offline store, join them to the orders the honest way, and train.

## **3. Setup**

The class needs three things in AWS. A **domain** is what SageMaker Studio runs in. A **space** is the machine your JupyterLab runs on. A **bucket** holds the offline store and the training files. Quick setup makes the domain and the bucket in under a minute.

### 3.1 Quick setup

Quick setup puts the domain in the account's **default VPC**, the network every account starts with. If someone deleted it, quick setup stops with an error.

**[Please Note:]** Before class. Paste the following service name into the AWS console search bar.


In [ ]:
VPC

Open **Your VPCs**. If the list is empty, choose **Actions → Create default VPC** and confirm. Check that the region at the top right says **United States (N. Virginia)**. Its region code is below.


In [ ]:
us-west-2

**[Please Note:]** Browser. Paste the following service name into the AWS console search bar and open it.


In [ ]:
SageMaker AI

In the left menu, under **Environment configuration**, click **Domains**, then **Create domain**. Keep **Set up for single user (Quick setup)** and click **Set up**. If the account already has a domain, skip this and use that one.

![c13_quick_setup.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/452/original/c13_quick_setup.png?1793273316)

The domain was ready 18 seconds after the click. Quick setup made three things you will use today: the domain itself, an execution role, and a bucket that the role is already allowed to use.

The role is who your code acts as. Its name starts with the following prefix, followed by a date. This prefix helps you recognise the generated role; the Python setup cell below discovers the complete role automatically.


In [ ]:
AmazonSageMaker-ExecutionRole-

The bucket's name starts with the following prefix, followed by your account number. The Python setup cell below builds the complete bucket name automatically.


In [ ]:
sagemaker-us-west-2-

### 3.2 A JupyterLab space

![c13_domain_ready.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/453/original/c13_domain_ready.png?1793273319)

**[Please Note:]** Browser. Click the domain's name in the **Domains** list and open the **User profiles** tab. Find the profile quick setup made. Its generated name starts with the following prefix; use the actual profile shown in your account.


In [ ]:
default-

On that profile's row, click **Open Studio**. Studio opens in a new browser tab. Skip the tour if it offers one. In Studio, click **JupyterLab** at the top left, then **Create JupyterLab space**. Paste the following into the space-name field.


In [ ]:
features

Click **Create space**. Keep the default instance and image, and click **Run space**. When the status says **Running**, click **Open JupyterLab**.

![c13_space_running.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/454/original/c13_space_running.png?1793273321)

### 3.3 Open this notebook in JupyterLab

**[Please Note:]** JupyterLab. Click the **Upload Files** button (the arrow pointing up) at the top of the file browser on the left, and choose this notebook from your laptop. Double-click it to open it. If JupyterLab asks for a kernel, pick **Python 3 (ipykernel)**. From here on, run the Python steps in this notebook. The left bar has a **Table of Contents** button that lists these sections.

![c13_notebook_open.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/455/original/c13_notebook_open.png?1793273323)

The first Python cell sets the names the class uses and finds the bucket and the role quick setup made.

**[Please Note:]** JupyterLab. Run the following Python cell.


In [1]:
"""The names this notebook uses, and the bucket and role quick setup made."""
import boto3
from sagemaker.core.helper.session_helper import get_execution_role

REGION = "us-west-2"
FEATURE_GROUP = "returns-customers"
ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
BUCKET = f"sagemaker-{REGION}-{ACCOUNT}"   # the bucket quick setup made
ROLE = get_execution_role()                # the role this notebook runs as

print("bucket:", BUCKET)
print("role:  ", ROLE.split("/")[-1])

bucket: sagemaker-us-west-2-245713705525
role:   AmazonSageMaker-ExecutionRole-20261104T211065


**Example output — saved from the original run.**

<pre>sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
bucket: sagemaker-us-west-2-111122223333
role:   AmazonSageMaker-ExecutionRole-20261029T094482
</pre>


Both names come back, so the notebook can reach AWS as the role. Every call from here on is made by that role, with the permissions quick setup gave it. The two `INFO` lines above them come from the SageMaker library, saying it found no settings file to load. They are harmless and appear every time it is imported.

## **4. From orders to feature rows**

The orders file sits on the course's file server. pandas reads it straight from there. The cell prints a short summary and C-00001's first four orders, the customer from section 1.

**[Please Note:]** JupyterLab. Run this cell.

In [2]:
"""Read the raw orders. Each row is one order and whether it came back."""
import pandas as pd

ORDERS_URL = "https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/246/719/original/returns_events.csv?1791977425"
orders = pd.read_csv(ORDERS_URL)

print(len(orders), "orders from", orders.customer_id.nunique(), "customers,", orders.order_date.min(), "to", orders.order_date.max())
print("share returned:", round(orders.returned.mean(), 3))
orders[orders.customer_id == "C-00001"].head(4)

12000 orders from 1440 customers, 2025-10-01 to 2026-09-30
share returned: 0.331


,order_id,customer_id,order_date,category,price,discount_pct,delivery_days,payment_method,is_gift,seller_rating,returned,signup_date
402,ORD-200402,C-00001,2025-10-13,apparel,1481.0,18,3,card,0,4.0,1,2023-08-15
528,ORD-200528,C-00001,2025-10-17,home,3164.0,37,2,cod,0,3.9,0,2023-08-15
2200,ORD-202200,C-00001,2025-12-07,apparel,684.0,16,1,upi,0,4.5,1,2023-08-15
2684,ORD-202684,C-00001,2025-12-22,books,1179.0,42,9,upi,0,3.8,1,2023-08-15


**Example output — saved from the original run.**

<pre>12000 orders from 1440 customers, 2025-10-01 to 2026-09-30
share returned: 0.331
</pre>

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>order_id</th>
      <th>customer_id</th>
      <th>order_date</th>
      <th>category</th>
      <th>price</th>
      <th>discount_pct</th>
      <th>delivery_days</th>
      <th>payment_method</th>
      <th>is_gift</th>
      <th>seller_rating</th>
      <th>returned</th>
      <th>signup_date</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>402</th>
      <td>ORD-200402</td>
      <td>C-00001</td>
      <td>2025-10-13</td>
      <td>apparel</td>
      <td>1481.0</td>
      <td>18</td>
      <td>3</td>
      <td>card</td>
      <td>0</td>
      <td>4.0</td>
      <td>1</td>
      <td>2023-08-15</td>
    </tr>
    <tr>
      <th>528</th>
      <td>ORD-200528</td>
      <td>C-00001</td>
      <td>2025-10-17</td>
      <td>home</td>
      <td>3164.0</td>
      <td>37</td>
      <td>2</td>
      <td>cod</td>
      <td>0</td>
      <td>3.9</td>
      <td>0</td>
      <td>2023-08-15</td>
    </tr>
    <tr>
      <th>2200</th>
      <td>ORD-202200</td>
      <td>C-00001</td>
      <td>2025-12-07</td>
      <td>apparel</td>
      <td>684.0</td>
      <td>16</td>
      <td>1</td>
      <td>upi</td>
      <td>0</td>
      <td>4.5</td>
      <td>1</td>
      <td>2023-08-15</td>
    </tr>
    <tr>
      <th>2684</th>
      <td>ORD-202684</td>
      <td>C-00001</td>
      <td>2025-12-22</td>
      <td>books</td>
      <td>1179.0</td>
      <td>42</td>
      <td>9</td>
      <td>upi</td>
      <td>0</td>
      <td>3.8</td>
      <td>1</td>
      <td>2023-08-15</td>
    </tr>
  </tbody>
</table>


C-00001 joined the shop in 2023. Their first order in this data was placed on 13 October 2025, and it came back. So did the one on 7 December.

### 4.1 One row every time a customer's history changes

A customer's history changes at two kinds of moment. When they place an order, their order count goes up by one. When a return becomes known, 14 days after the order, their return count goes up by one. The next cell writes one **feature row** for every such moment, holding the customer's totals as of that moment.

Each row has four features.

| Feature | Meaning |
|---|---|
| `orders_so_far` | Orders placed up to and including this moment |
| `returns_so_far` | Returns known up to this moment |
| `return_rate` | Returns over orders, pulled toward 20% while the history is short |
| `tenure_days` | Days since the customer signed up |

The pull toward 20% stops one order from making a customer look like a 0% or a 100% returner. The formula is below.


In [3]:
(returns + 1) / (orders + 5)

NameError: name 'returns' is not defined

| Orders | Returns | Actual return rate | Smoothed return rate |
|---:|---:|---:|---:|
| 1 | 0 | 0% | 16.67% |
| 5 | 0 | 0% | 10.00% |
| 10 | 0 | 0% | 6.67% |
| 100 | 0 | 0% | 0.95% |
| 10 | 5 | 50.00% | 40.00% |
| 100 | 50 | 50.00% | 48.57% |

Each row also carries `event_time`, the moment it describes, written the way Feature Store expects. Here is an example of the timestamp format.


In [ ]:
2025-10-13T00:00:00Z

**[Please Note:]** JupyterLab. Run this cell. Walk through `customer_features` top to bottom before running it, using the two moments described above.


In [6]:
df = orders.copy()
df["order_date"] = pd.to_datetime(df["order_date"])

# Step 1: Create an event whenever an order is placed
order_events = pd.DataFrame({
    "customer_id": df["customer_id"],
    "time": df["order_date"],
    "new_orders": 1,
    "new_returns": 0
})

# Step 2: Create an event when a return becomes known
returned_orders = df[df["returned"] == 1]

return_events = pd.DataFrame({
    "customer_id": returned_orders["customer_id"],
    "time": returned_orders["order_date"] + RETURN_KNOWN_AFTER,
    "new_orders": 0,
    "new_returns": 1
})

# Step 3: Combine both types of events
all_events = pd.concat(
    [order_events, return_events],
    ignore_index=True
)

# Step 4: Combine events occurring at the same time
all_events = (
    all_events
    .groupby(["customer_id", "time"], as_index=False)
    [["new_orders", "new_returns"]]
    .sum()
)

all_events = all_events.sort_values(
    ["customer_id", "time"]
)

# Step 5: Calculate cumulative customer statistics
customer_groups = all_events.groupby("customer_id")

all_events["orders_so_far"] = customer_groups["new_orders"].cumsum()
all_events["returns_so_far"] = customer_groups["new_returns"].cumsum()
all_events.loc[all_events['customer_id'] == 'C-00001'].head(10)

,customer_id,time,new_orders,new_returns,orders_so_far,returns_so_far
0,C-00001,2025-10-13,1,0,1,0
1,C-00001,2025-10-17,1,0,2,0
2,C-00001,2025-10-27,0,1,2,1
3,C-00001,2025-12-07,1,0,3,1
4,C-00001,2025-12-21,0,1,3,2
5,C-00001,2025-12-22,1,0,4,2
6,C-00001,2026-01-05,0,1,4,3
7,C-00001,2026-03-01,1,0,5,3
8,C-00001,2026-03-15,0,1,5,4
9,C-00001,2026-04-04,1,0,6,4


In [7]:
"""Turn the orders into customer feature rows: a new row each time a customer's history changes."""
RETURN_KNOWN_AFTER = pd.Timedelta(days=14)   # a returned parcel takes two weeks to come back

def customer_features(orders):
    df = orders.copy()
    df["order_date"] = pd.to_datetime(df["order_date"])

    # Step 1: Create an event whenever an order is placed
    order_events = pd.DataFrame({
        "customer_id": df["customer_id"],
        "time": df["order_date"],
        "new_orders": 1,
        "new_returns": 0
    })

    # Step 2: Create an event when a return becomes known
    returned_orders = df[df["returned"] == 1]

    return_events = pd.DataFrame({
        "customer_id": returned_orders["customer_id"],
        "time": returned_orders["order_date"] + RETURN_KNOWN_AFTER,
        "new_orders": 0,
        "new_returns": 1
    })

    # Step 3: Combine both types of events
    all_events = pd.concat(
        [order_events, return_events],
        ignore_index=True
    )

    # Step 4: Combine events occurring at the same time
    all_events = (
        all_events
        .groupby(["customer_id", "time"], as_index=False)
        [["new_orders", "new_returns"]]
        .sum()
    )

    all_events = all_events.sort_values(
        ["customer_id", "time"]
    )

    # Step 5: Calculate cumulative customer statistics
    customer_groups = all_events.groupby("customer_id")

    all_events["orders_so_far"] = customer_groups["new_orders"].cumsum()
    all_events["returns_so_far"] = customer_groups["new_returns"].cumsum()

    all_events["return_rate"] = (
        (all_events["returns_so_far"] + 1) /
        (all_events["orders_so_far"] + 5)
    ).round(4)

    # Step 6: Calculate customer tenure
    signup_dates = pd.to_datetime(
        df.groupby("customer_id")["signup_date"].first()
    )

    all_events["signup_date"] = all_events["customer_id"].map(signup_dates)

    all_events["tenure_days"] = (
        all_events["time"] - all_events["signup_date"]
    ).dt.days

    # Step 7: Format event timestamps for SageMaker
    all_events["event_time"] = all_events["time"].dt.strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )

    # Step 8: Return only the features needed by Feature Store
    feature_columns = [
        "customer_id",
        "event_time",
        "orders_so_far",
        "returns_so_far",
        "return_rate",
        "tenure_days"
    ]

    return all_events[feature_columns].reset_index(drop=True)



rows = customer_features(orders)
print(len(rows), "feature rows")
rows[rows.customer_id == "C-00001"].head(10)


15598 feature rows


,customer_id,event_time,orders_so_far,returns_so_far,return_rate,tenure_days
0,C-00001,2025-10-13T00:00:00Z,1,0,0.1667,790
1,C-00001,2025-10-17T00:00:00Z,2,0,0.1429,794
2,C-00001,2025-10-27T00:00:00Z,2,1,0.2857,804
3,C-00001,2025-12-07T00:00:00Z,3,1,0.2500,845
4,C-00001,2025-12-21T00:00:00Z,3,2,0.3750,859
5,C-00001,2025-12-22T00:00:00Z,4,2,0.3333,860
6,C-00001,2026-01-05T00:00:00Z,4,3,0.4444,874
7,C-00001,2026-03-01T00:00:00Z,5,3,0.4000,929
8,C-00001,2026-03-15T00:00:00Z,5,4,0.5000,943
9,C-00001,2026-04-04T00:00:00Z,6,4,0.4545,963


**Example output — saved from the original run.**

<pre>15598 feature rows
</pre>

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>customer_id</th>
      <th>event_time</th>
      <th>orders_so_far</th>
      <th>returns_so_far</th>
      <th>return_rate</th>
      <th>tenure_days</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>C-00001</td>
      <td>2025-10-13T00:00:00Z</td>
      <td>1</td>
      <td>0</td>
      <td>0.1667</td>
      <td>790</td>
    </tr>
    <tr>
      <th>1</th>
      <td>C-00001</td>
      <td>2025-10-17T00:00:00Z</td>
      <td>2</td>
      <td>0</td>
      <td>0.1429</td>
      <td>794</td>
    </tr>
    <tr>
      <th>2</th>
      <td>C-00001</td>
      <td>2025-10-27T00:00:00Z</td>
      <td>2</td>
      <td>1</td>
      <td>0.2857</td>
      <td>804</td>
    </tr>
    <tr>
      <th>3</th>
      <td>C-00001</td>
      <td>2025-12-07T00:00:00Z</td>
      <td>3</td>
      <td>1</td>
      <td>0.2500</td>
      <td>845</td>
    </tr>
  </tbody>
</table>


Read C-00001's rows next to their orders above.

- **13 October.** The first order. One order, no returns yet, so the rate is (0 + 1) / (1 + 5) = 0.1667.
- **17 October.** A second order. Two orders, still no returns known.
- **27 October.** No order that day. It is 13 October plus 14 days, when the parcel from the first order arrived back. The return count goes up to 1.
- **7 December.** A third order.

The 12,000 orders became 15,598 rows, because returns add moments of their own. Every row is true from its `event_time` until the customer's next row. That is the shape a feature store wants.

## **5. A feature group**

A **feature group** is a table inside Feature Store. You tell it three things when you create it.

- **The columns and their types.** Feature Store knows three types, `String`, `Integral` for whole numbers and `Fractional` for decimals.
- **The record identifier**, the column that says whose row it is. Here that is `customer_id`.
- **The event time**, the column that says when the row became true. Here that is `event_time`.

It also needs to know where to keep the offline store, which is a folder in S3, and which role may write there. The next cell creates the group and waits until it is ready.

**[Please Note:]** JupyterLab. Run this cell. It takes about half a minute.

In [8]:
"""Create the feature group: its columns, which column names the customer, which holds the time."""
import time

sm = boto3.client("sagemaker")   # the client that creates and describes SageMaker things
COLUMNS = [("customer_id", "String"), ("event_time", "String"),
           ("orders_so_far", "Integral"), ("returns_so_far", "Integral"),
           ("return_rate", "Fractional"), ("tenure_days", "Integral")]

start = time.time()
try:
    sm.create_feature_group(
        FeatureGroupName=FEATURE_GROUP,
        RecordIdentifierFeatureName="customer_id",   # whose row it is
        EventTimeFeatureName="event_time",           # when it became true
        FeatureDefinitions=[{"FeatureName": name, "FeatureType": kind} for name, kind in COLUMNS],
        OfflineStoreConfig={"S3StorageConfig": {"S3Uri": f"s3://{BUCKET}/feature-store"}},
        RoleArn=ROLE,
    )
except sm.exceptions.ResourceInUse:
    print(FEATURE_GROUP, "already exists")

status = "Creating"
while status == "Creating":
    time.sleep(5)
    group = sm.describe_feature_group(FeatureGroupName=FEATURE_GROUP)
    status = group["FeatureGroupStatus"]

print("status:", status, "after", round(time.time() - start), "s")
print("offline store:", group["OfflineStoreConfig"]["S3StorageConfig"]["ResolvedOutputS3Uri"])
print("table:", group["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"])

status: Created after 22 s
offline store: s3://sagemaker-us-west-2-245713705525/feature-store/245713705525/sagemaker/us-west-2/offline-store/returns-customers-1793807793/data
table: returns_customers_1793807793


**Example output — saved from the original run.**

<pre>status: Created after 22 s
offline store: s3://sagemaker-us-west-2-111122223333/feature-store/111122223333/sagemaker/us-west-2/offline-store/returns-customers-1793270435/data
table: returns_customers_1793270435
</pre>


The group is `Created`. Two lines in the output are worth a look.

The **offline store** is a folder in the bucket, with your account number and the group's name added to the path. Rows will land there as files.

The **table** is an entry in the AWS Glue Data Catalog, which Feature Store made for you. It describes the files in that folder, their columns and types, so that a SQL engine can read them as a table. Section 7 uses it.

Studio has a page for feature groups too.

**[Please Note:]** Studio (the browser tab you opened JupyterLab from). In the left menu click **More**, then **Feature Store**. Find and open the feature group with the following name.


In [ ]:
returns-customers

Open the **Features** tab. Come back to JupyterLab afterwards.

![c13_studio_feature_group.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/456/original/c13_studio_feature_group.png?1793273326)

The **Features** tab lists the six columns with the types the cell gave them. Nothing on the page shows data, because none has been written yet.


## **6. Writing the rows**

Rows go in with one call each, `put_record`. The call belongs to a second client, `sagemaker-featurestore-runtime`, the one that reads and writes rows. The `sagemaker` client from the last cell manages groups but never touches rows.

A record is a list of name and value pairs, and every value travels as text. The group's column types decide how each value is stored. The cell first writes one row and shows it, then writes all of them, eight calls at a time so it takes seconds instead of minutes.

**[Please Note:]** JupyterLab. Run this cell. The first write is retried for up to two minutes, because a brand-new group can refuse writes for a few seconds after it says `Created`.

In [9]:
"""Write every feature row into the feature group, one call per row, eight calls at a time."""
from concurrent.futures import ThreadPoolExecutor

store = boto3.client("sagemaker-featurestore-runtime")   # the client that reads and writes rows


def put(row):
    record = [{"FeatureName": name, "ValueAsString": str(value)} for name, value in row.items()]
    store.put_record(FeatureGroupName=FEATURE_GROUP, Record=record)


first = rows.iloc[0].to_dict()
for attempt in range(24):
    try:
        put(first)
        break
    except store.exceptions.ResourceNotFound:
        time.sleep(5)
print("one row:", first)

start = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(put, rows.to_dict("records")))
print("wrote", len(rows), "rows in", round(time.time() - start), "s")

one row: {'customer_id': 'C-00001', 'event_time': '2025-10-13T00:00:00Z', 'orders_so_far': 1, 'returns_so_far': 0, 'return_rate': 0.1667, 'tenure_days': 790}
wrote 15598 rows in 25 s


**Example output — saved from the original run.**

<pre>one row: {&#x27;customer_id&#x27;: &#x27;C-00001&#x27;, &#x27;event_time&#x27;: &#x27;2025-10-13T00:00:00Z&#x27;, &#x27;orders_so_far&#x27;: 1, &#x27;returns_so_far&#x27;: 0, &#x27;return_rate&#x27;: 0.1667, &#x27;tenure_days&#x27;: 790}
wrote 15598 rows in 24 s
</pre>


In [ ]:
# records = rows.to_dict("records")

# for i in range(0, len(records), 25):
#     batch = records[i:i + 25]

#     entries = [
#         {
#             "FeatureGroupName": FEATURE_GROUP,
#             "Record": [
#                 {"FeatureName":k, "ValueAsString":str(v)}
#                 for k, v in row.items()
#             ]
#         }
#         for row in batch
#     ]

#     response = store.batch_write_record(Entries=entries)

#     if response.get("Errors") or response.get("UnprocessedEntries"):
#         raise RuntimeError(f"Batch incomplete: {response}")

# print(f"Processed {len(records)} records")

All 15,598 rows went in, in 24 seconds. The first row went in twice, once alone and once with the rest. That does no harm, because both copies are the same row.

## **7. The offline store**

Every row you write goes to the offline store, the folder in S3. It does not go there at once. Feature Store collects writes and saves them as files in batches, a few minutes later. Until then, the folder is empty.

To read the files you need a SQL engine. **Athena** is the AWS service that runs SQL over files in S3. It finds the columns in the Glue table from section 5, reads the files in the folder, and writes its answer as a CSV file back to S3. The next cell wraps that in a small function, `query`, that sends SQL and returns the answer as a pandas table.

**[Please Note:]** JupyterLab. Run this cell. It only defines the function.

In [10]:
"""Ask Athena a SQL question about the offline store and get the answer back as a table."""
athena = boto3.client("athena")
s3 = boto3.client("s3")
DATABASE = group["OfflineStoreConfig"]["DataCatalogConfig"]["Database"]
TABLE = group["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"]


def query(sql):
    run = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE},
        ResultConfiguration={"OutputLocation": f"s3://{BUCKET}/athena/"},   # where Athena writes the answer
    )
    while True:
        state = athena.get_query_execution(QueryExecutionId=run["QueryExecutionId"])["QueryExecution"]["Status"]
        if state["State"] not in ("QUEUED", "RUNNING"):
            break
        time.sleep(1)
    if state["State"] != "SUCCEEDED":
        raise RuntimeError(state.get("StateChangeReason", state["State"]))
    answer = s3.get_object(Bucket=BUCKET, Key=f"athena/{run['QueryExecutionId']}.csv")
    return pd.read_csv(answer["Body"])

Now count the rows in the offline store every 30 seconds until all of them are there.

**[Please Note:]** JupyterLab. Run this cell. It took about six minutes in the recorded run. While it waits, read section 8.1 with the class, which needs no code.

In [12]:
"""Watch the offline store fill up: count its rows every 30 seconds until all of them are there."""
start = time.time()
count = 0
while count < len(rows) and time.time() - start < 20 * 60:
    time.sleep(30)
    count = query(f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0]
    print(f"{time.time() - start:4.0f} s   rows in the offline store: {count}")

query(f"""SELECT customer_id, event_time, orders_so_far, returns_so_far, return_rate,
                 write_time, api_invocation_time, is_deleted
          FROM {TABLE} WHERE customer_id = 'C-00001' ORDER BY event_time, write_time LIMIT 5""")

  31 s   rows in the offline store: 0
  64 s   rows in the offline store: 0
  95 s   rows in the offline store: 0
 126 s   rows in the offline store: 0
 160 s   rows in the offline store: 15599


,customer_id,event_time,orders_so_far,returns_so_far,return_rate,write_time,api_invocation_time,is_deleted
0,C-00001,2025-10-13T00:00:00Z,1,0,0.1667,2026-11-04 16:02:41.601,2026-11-04 15:57:06.000,False
1,C-00001,2025-10-13T00:00:00Z,1,0,0.1667,2026-11-04 16:02:41.601,2026-11-04 15:57:07.000,False
2,C-00001,2025-10-17T00:00:00Z,2,0,0.1429,2026-11-04 16:02:41.620,2026-11-04 15:57:07.000,False
3,C-00001,2025-10-27T00:00:00Z,2,1,0.2857,2026-11-04 16:02:40.927,2026-11-04 15:57:07.000,False
4,C-00001,2025-12-07T00:00:00Z,3,1,0.2500,2026-11-04 16:02:40.775,2026-11-04 15:57:07.000,False


**Example output — saved from the original run.**

<pre>  31 s   rows in the offline store: 0
  62 s   rows in the offline store: 0
  94 s   rows in the offline store: 0
 125 s   rows in the offline store: 0
 156 s   rows in the offline store: 0
 187 s   rows in the offline store: 0
 219 s   rows in the offline store: 0
 250 s   rows in the offline store: 0
 281 s   rows in the offline store: 0
 312 s   rows in the offline store: 0
 346 s   rows in the offline store: 15598
</pre>

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>customer_id</th>
      <th>event_time</th>
      <th>orders_so_far</th>
      <th>returns_so_far</th>
      <th>return_rate</th>
      <th>write_time</th>
      <th>api_invocation_time</th>
      <th>is_deleted</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>C-00001</td>
      <td>2025-10-13T00:00:00Z</td>
      <td>1</td>
      <td>0</td>
      <td>0.1667</td>
      <td>2026-10-29 10:47:02.941</td>
      <td>2026-10-29 10:40:59.000</td>
      <td>False</td>
    </tr>
    <tr>
      <th>1</th>
      <td>C-00001</td>
      <td>2025-10-17T00:00:00Z</td>
      <td>2</td>
      <td>0</td>
      <td>0.1429</td>
      <td>2026-10-29 10:47:02.948</td>
      <td>2026-10-29 10:40:59.000</td>
      <td>False</td>
    </tr>
    <tr>
      <th>2</th>
      <td>C-00001</td>
      <td>2025-10-27T00:00:00Z</td>
      <td>2</td>
      <td>1</td>
      <td>0.2857</td>
      <td>2026-10-29 10:47:02.246</td>
      <td>2026-10-29 10:40:59.000</td>
      <td>False</td>
    </tr>
    <tr>
      <th>3</th>
      <td>C-00001</td>
      <td>2025-12-07T00:00:00Z</td>
      <td>3</td>
      <td>1</td>
      <td>0.2500</td>
      <td>2026-10-29 10:47:02.108</td>
      <td>2026-10-29 10:40:59.000</td>
      <td>False</td>
    </tr>
    <tr>
      <th>4</th>
      <td>C-00001</td>
      <td>2025-12-21T00:00:00Z</td>
      <td>3</td>
      <td>2</td>
      <td>0.3750</td>
      <td>2026-10-29 10:47:02.857</td>
      <td>2026-10-29 10:40:59.000</td>
      <td>False</td>
    </tr>
  </tbody>
</table>


For more than five minutes the count stayed at 0. Then, at 346 seconds, all 15,598 rows appeared at once. Feature Store had collected the writes and saved them to S3 together.

The table at the end is C-00001's first five rows, read back from S3, the same rows section 4 computed. Feature Store added three columns of its own. `api_invocation_time` is when your `put_record` call arrived. `write_time` is when the row was saved to the offline store, about six minutes later here. `is_deleted` marks rows removed with a delete call.

The offline store keeps every row you write. It never replaces a customer's older row with a newer one, so all 25 of C-00001's rows are there. That history is exactly what the next section needs.

## **8. A training set that only uses the past**

### 8.1 Which row belongs to which order

To train, every order needs its customer's features. C-00001 has 25 feature rows. Which one belongs to their order of 10 August 2026? Here are three of them.


In [ ]:
event_time   orders_so_far  returns_so_far  return_rate  what happened that day
2026-06-27              11               9       0.6250  an order
2026-08-10              12               9       0.5882  the 10 August order itself
2026-08-24              12              10       0.6471  the 10 August order came back

The right row is the newest one from **strictly before** 10 August, the one from 27 June. That is everything the shop knew when the order was placed. The row from 10 August already counts the order being scored. The row from 24 August contains the answer the model is supposed to guess, because that is the day the 10 August order came back.

A simple join that gives every order the customer's newest row picks the row from 14 September, the last one. It counts 14 orders and 11 returns, including the return of this order. For the customer's very first order, in October 2025, it would say they had returned 11 of 14 orders before buying anything.

Picking the newest row from before each order's date, for every order, is called a **point-in-time join**. It is only possible because the offline store kept every row with its event time.

### 8.2 Both joins, side by side

The next cell reads every feature row back from the offline store. Then it builds two tables of orders with features. The honest one uses `pd.merge_asof`, which finds, for each order, the customer's newest row before the order date. `allow_exact_matches=False` makes it strictly before. The naive one gives every order the customer's newest row of all. It prints the 10 August order from both.

**[Please Note:]** JupyterLab. Run this cell.


In [ ]:
#"""Two ways to give each order its customer's features. Only the first is honest."""
#FEATS = ["orders_so_far", "returns_so_far", "return_rate", "tenure_days"]
#feats = query(f"SELECT customer_id, event_time, {', '.join(FEATS)} FROM {TABLE}")
#feats["time"] = pd.to_datetime(feats.event_time).dt.tz_localize(None)
#feats = feats.sort_values("time")
#o = orders.assign(order_date=pd.to_datetime(orders.order_date)).sort_values("order_date")

## Honest: the customer's newest row from strictly before the order date.
#honest = pd.merge_asof(o, feats[["customer_id", "time"] + FEATS], by="customer_id",
#                       left_on="order_date", right_on="time", allow_exact_matches=False)
## Naive: the customer's newest row of all, whenever it was written.
#latest = feats.groupby("customer_id").tail(1)
#naive = o.merge(latest[["customer_id", "time"] + FEATS], on="customer_id", how="left")

#show = ["order_id", "order_date", "returned", "time"] + FEATS
#print("honest:\n", honest[honest.order_id == "ORD-210337"][show].to_string(index=False))
#print("naive:\n", naive[naive.order_id == "ORD-210337"][show].to_string(index=False))
#print("orders with no feature row before them:", honest.orders_so_far.isna().sum())

In [ ]:


date = pd.to_datetime("2026-01-10T15:30:00Z")

print(date)
# 2026-01-10 15:30:00+00:00

print(date.tz_localize(None))
# 2026-01-10 15:30:00

In [13]:
FEATS = ["orders_so_far", "returns_so_far", "return_rate", "tenure_days"]

# Load historical customer features from Athena
feats = query(f"SELECT customer_id, event_time, {', '.join(FEATS)} FROM {TABLE}")

feats.head()

,customer_id,event_time,orders_so_far,returns_so_far,return_rate,tenure_days
0,C-00516,2025-10-17T00:00:00Z,2,2,0.4286,402
1,C-01476,2025-10-11T00:00:00Z,3,0,0.1250,93
2,C-01294,2025-10-06T00:00:00Z,1,0,0.1667,876
3,C-01042,2025-10-09T00:00:00Z,1,0,0.1667,583
4,C-00365,2025-10-18T00:00:00Z,2,0,0.1429,156


In [ ]:
# 2025-10-18T00:00:00Z -> UTC Time Zone (Z -> Zero diff from UTC Time Zone)
# 2025-10-18T00:00:00+5:30 -> IST

In [14]:
feats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15599 entries, 0 to 15598
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     15599 non-null  object 
 1   event_time      15599 non-null  object 
 2   orders_so_far   15599 non-null  int64  
 3   returns_so_far  15599 non-null  int64  
 4   return_rate     15599 non-null  float64
 5   tenure_days     15599 non-null  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 731.3+ KB


In [18]:
# Convert dates and sort
feats["event_time"] = pd.to_datetime(feats["event_time"], utc=True).dt.tz_localize(None)
feats = feats.sort_values("event_time")
feats.head()

,customer_id,event_time,orders_so_far,returns_so_far,return_rate,tenure_days
865,C-00262,2025-10-01,1,0,0.1667,417
1088,C-01426,2025-10-01,1,0,0.1667,761
252,C-00721,2025-10-01,1,0,0.1667,107
254,C-00474,2025-10-01,1,0,0.1667,453
800,C-00602,2025-10-01,1,0,0.1667,542


In [19]:
feats.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15599 entries, 865 to 14646
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customer_id     15599 non-null  object        
 1   event_time      15599 non-null  datetime64[ns]
 2   orders_so_far   15599 non-null  int64         
 3   returns_so_far  15599 non-null  int64         
 4   return_rate     15599 non-null  float64       
 5   tenure_days     15599 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(1)
memory usage: 853.1+ KB


In [20]:
o = orders.copy()
o["order_date"] = pd.to_datetime(o["order_date"])
o = o.sort_values("order_date")
o.head()

,order_id,customer_id,order_date,category,price,discount_pct,delivery_days,payment_method,is_gift,seller_rating,returned,signup_date
0,ORD-200000,C-00029,2025-10-01,home,344.0,3,2,cod,0,3.1,0,2023-09-27
31,ORD-200031,C-01470,2025-10-01,home,3691.0,26,6,card,0,4.1,0,2023-05-01
30,ORD-200030,C-01426,2025-10-01,books,900.0,22,2,card,1,4.5,0,2023-09-01
29,ORD-200029,C-01363,2025-10-01,apparel,339.0,6,2,wallet,0,4.9,0,2024-12-20
28,ORD-200028,C-01334,2025-10-01,apparel,5460.0,40,2,upi,0,3.3,0,2023-11-02


In [21]:



# Honest: latest customer history strictly before each order
honest = pd.merge_asof(
    o,
    feats,
    by="customer_id",
    left_on="order_date",
    right_on="event_time",
    direction="backward",
    allow_exact_matches=False
)

# Naive: latest customer history regardless of order date
latest = feats.groupby("customer_id").tail(1)
naive = o.merge(latest, on="customer_id", how="left")

# Compare results for one order
columns = ["order_id", "order_date", "returned", "event_time"] + FEATS

for name, df in [("Honest", honest), ("Naive", naive)]:
    print(f"\n{name}:")
    print(
        df.loc[df["order_id"] == "ORD-210337", columns]
        .to_string(index=False)
    )

print("\nOrders without previous history:", honest["orders_so_far"].isna().sum())


Honest:
  order_id order_date  returned event_time  orders_so_far  returns_so_far  return_rate  tenure_days
ORD-210337 2026-08-10         1 2026-06-27           11.0             9.0        0.625       1047.0

Naive:
  order_id order_date  returned event_time  orders_so_far  returns_so_far  return_rate  tenure_days
ORD-210337 2026-08-10         1 2026-09-14             14              11       0.6316         1126

Orders without previous history: 1456


**Example output — saved from the original run.**

<pre>honest:
   order_id order_date  returned       time  orders_so_far  returns_so_far  return_rate  tenure_days
ORD-210337 2026-08-10         1 2026-06-27           11.0             9.0        0.625       1047.0
naive:
   order_id order_date  returned       time  orders_so_far  returns_so_far  return_rate  tenure_days
ORD-210337 2026-08-10         1 2026-09-14             14              11       0.6316         1126
orders with no feature row before them: 1456
</pre>


The honest table gives the 10 August order the row from 27 June. The naive one gives it the row from 14 September, a month after the order, with its own return already counted.

1,456 orders have no row before them at all. They are each customer's first order of the year, placed before anything about them was recorded. The next cell fills them the way a brand-new customer looks, with no orders, no returns, a 20% rate and the tenure worked out from the signup date.

### 8.3 What the wrong join costs

The next cell lays both tables out the way XGBoost wants them, the label first and then 18 numbers in a fixed order. The order columns, the four features, and one 0-or-1 column for each category and each payment method. Then it trains a quick model on each table, right here in the notebook, on orders before July. And it scores them on orders from July on, the way a model is checked before it goes live.

**[Please Note:]** JupyterLab. Run this cell. It takes a few seconds.

In [22]:
"""Lay out both tables for XGBoost, train a quick model on each, and score them on orders from July on."""
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

ORDER_COLS = ["price", "discount_pct", "delivery_days", "is_gift", "seller_rating"]
CATEGORIES = ["apparel", "beauty", "books", "electronics", "home"]
PAYMENTS = ["card", "cod", "upi", "wallet"]
CUT = pd.Timestamp("2026-07-01")   # train on orders before this day, check on orders from it on


def model_table(d):
    """The label first, then the 18 numbers the model reads, always in this order."""
    d = d.fillna({"orders_so_far": 0, "returns_so_far": 0, "return_rate": 0.2})   # no history yet
    d["tenure_days"] = d.tenure_days.fillna((d.order_date - pd.to_datetime(d.signup_date)).dt.days)
    t = d[["returned"] + ORDER_COLS + FEATS].copy()
    for c in CATEGORIES:
        t[f"category_{c}"] = (d.category == c).astype(int)
    for p in PAYMENTS:
        t[f"payment_{p}"] = (d.payment_method == p).astype(int)
    return t


H, N = model_table(honest), model_table(naive)   # both keep the orders in the same order
train, check = honest.order_date < CUT, honest.order_date >= CUT


def quick_model(t):
    return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1).fit(t[train].iloc[:, 1:], t[train].returned)


def auc(model, t):
    return roc_auc_score(t[check].returned, model.predict_proba(t[check].iloc[:, 1:])[:, 1])


honest_model, naive_model = quick_model(H), quick_model(N)
print(f"honest model, checked on honest rows: {auc(honest_model, H):.3f}")
print(f"naive model,  checked on naive rows:  {auc(naive_model, N):.3f}")
print(f"naive model,  checked on honest rows: {auc(naive_model, H):.3f}")

honest model, checked on honest rows: 0.715
naive model,  checked on naive rows:  0.833
naive model,  checked on honest rows: 0.714


**Example output — saved from the original run.**

<pre>honest model, checked on honest rows: 0.715
naive model,  checked on naive rows:  0.833
naive model,  checked on honest rows: 0.714
</pre>


Read the three lines in order.

The naive model, checked on its own kind of rows, scores 0.833. That is the number a team would report, and it looks like a better model than the honest one at 0.715.

It is not. Live, at checkout, nobody has the customer's future rows. The model only ever gets the history up to that moment, which is exactly what the honest rows hold. Checked on those, the naive model scores 0.714, about the same as the honest model. The naive model was never better. Only its test was wrong. The checking rows contained returns that had not happened yet when the orders were placed, and the model read the answers from them. The 0.833 would have been reported, and checkout would never have seen it.

Nothing failed and nothing warned. The only defence is the join, and the join is only possible because the offline store keeps every row with its time.

### **Checkpoint 8.1**

Pick another customer, such as the following customer ID.


In [ ]:
C-00002

Print their orders and their honest rows side by side, using `orders` and `honest` from the cells above. For two of their orders, check by hand that the feature row's `time` is before the order date and that `orders_so_far` counts only the earlier orders. **Why it matters.** This is the check to run every time a training set is built from a feature store, because a wrong join gives no error, only a score that is too good.


In [ ]:

#o = orders.copy()
#o["order_date"] = pd.to_datetime(o["order_date"])
#o = o.sort_values("order_date")

#ORDER_COLS = ["price", "discount_pct", "delivery_days", "is_gift", "seller_rating"]
#CATEGORIES = ["apparel", "beauty", "books", "electronics", "home"]
#PAYMENTS = ["card", "cod", "upi", "wallet"]

#H = o[["returned"] + ORDER_COLS].copy()

#for c in CATEGORIES:
#    H[f"category_{c}"] = (o.category == c).astype(int)

#for p in PAYMENTS:
#    H[f"payment_{p}"] = (o.payment_method == p).astype(int)

#CUT = pd.Timestamp("2026-07-01")

#train = o.order_date < CUT
#check = o.order_date >= CUT

#print("Training:", H[train].shape)
#print("Validation:", H[check].shape)


## **9. Training on SageMaker**

The quick models were a measuring stick. The model the shop will serve is trained the way the SageMaker classes trained one, as a **training job** with the built-in XGBoost container. The job leaves a `model.tar.gz` in S3 that an endpoint can load, and the next class does exactly that.

The job reads CSV files from S3 with no header and the label in the first column, which is the layout `model_table` already made. The next cell saves the honest table as two files, orders before July for training and from July on for validation.

**[Please Note:]** JupyterLab. Run this cell.

In [23]:
"""Save the honest table as the two files the training job reads: no header, label first."""
s3.put_object(Bucket=BUCKET, Key="fs-train/train.csv", Body=H[train].to_csv(header=False, index=False))
s3.put_object(Bucket=BUCKET, Key="fs-validation/validation.csv", Body=H[check].to_csv(header=False, index=False))
print("train:", H[train].shape, "  validation:", H[check].shape)

train: (9014, 19)   validation: (2986, 19)


**Example output — saved from the original run.**

<pre>train: (9014, 19)   validation: (2986, 19)
</pre>


Now the training job. One call, `create_training_job`, describes all of it.

- **The image.** SageMaker's built-in XGBoost container, the same one the SageMaker classes used.
- **The role** the job runs as, so it can read and write the bucket.
- **The hyperparameters**, the same settings as the quick honest model, so its validation AUC should land close to it. SageMaker passes them to the container as text.
- **The two input channels**, `train` and `validation`, each pointing at a folder of CSV files.
- **Where to write the model**, and **the machine** to train on.

The cell then checks the job's status every 30 seconds until it is no longer `InProgress`, and prints what the job reported.

**[Please Note:]** JupyterLab. Run this cell. It takes about four minutes, most of it waiting for a machine. The job also appears in the SageMaker AI console under **Model training & customization → Training & tuning jobs** if you want to show it there.

In [ ]:
"""Train the honest model as a SageMaker training job with the built-in XGBoost container."""
XGBOOST_IMAGE = "683313688378.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.7-1"   # SageMaker's XGBoost, us-west-2
job_name = "returns-fs-" + time.strftime("%Y%m%d-%H%M%S")

sm.create_training_job(
    TrainingJobName=job_name,
    AlgorithmSpecification={"TrainingImage": XGBOOST_IMAGE, "TrainingInputMode": "File"},
    RoleArn=ROLE,
    HyperParameters={"objective": "binary:logistic", "eval_metric": "auc",
                     "num_round": "200", "eta": "0.1", "max_depth": "4"},   # the quick honest model's settings
    InputDataConfig=[
        {"ChannelName": "train", "ContentType": "text/csv",
         "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{BUCKET}/fs-train/"}}},
        {"ChannelName": "validation", "ContentType": "text/csv",
         "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{BUCKET}/fs-validation/"}}},
    ],
    OutputDataConfig={"S3OutputPath": f"s3://{BUCKET}/fs-output/"},
    ResourceConfig={"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 5},
    StoppingCondition={"MaxRuntimeInSeconds": 3600},
)

start = time.time()
status = "InProgress"
while status == "InProgress":
    time.sleep(30)
    job = sm.describe_training_job(TrainingJobName=job_name)
    status = job["TrainingJobStatus"]

print("job:", job_name, status, "after", round(time.time() - start), "s")
print("validation AUC:", round([m["Value"] for m in job["FinalMetricDataList"] if m["MetricName"] == "validation:auc"][0], 3))
print("model:", job["ModelArtifacts"]["S3ModelArtifacts"])

**Example output — saved from the original run.**

<pre>job: returns-fs-20261029-112411 Completed after 180 s
validation AUC: 0.721
model: s3://sagemaker-us-west-2-111122223333/fs-output/returns-fs-20261029-112411/output/model.tar.gz
</pre>


The job finished after 180 seconds with a validation AUC of 0.721, close to the quick honest model's 0.715. The small difference comes from the two running different versions of XGBoost. The last line is where the job left the trained model, a `model.tar.gz` file.

That model was trained only on what the shop knew at the time of each order, so its score is one the shop can expect at checkout. Every feature it used came out of the offline store.

## **10. Tear down**

An idle feature group costs nothing beyond its few megabytes in S3. Still, clean up what the class made. The next class starts from a fresh feature group.

**[Please Note:]** JupyterLab. Run this cell.

In [ ]:
"""Delete the feature group, then the files it and this notebook left in the bucket."""
try:
    sm.delete_feature_group(FeatureGroupName=FEATURE_GROUP)
    print("deleted feature group", FEATURE_GROUP)
except sm.exceptions.ResourceNotFound:
    print("no feature group", FEATURE_GROUP)

for prefix in ["feature-store/", "athena/", "fs-train/", "fs-validation/", "fs-output/"]:
    keys = [item["Key"] for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix)
            for item in page.get("Contents", [])]
    for i in range(0, len(keys), 1000):   # one delete call takes at most 1,000 files
        s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": k} for k in keys[i:i + 1000]]})
    print("removed", len(keys), "files under", prefix)

**Example output — saved from the original run.**

<pre>deleted feature group returns-customers
removed 10865 files under feature-store/
removed 28 files under athena/
removed 1 files under fs-train/
removed 1 files under fs-validation/
removed 2 files under fs-output/
</pre>


The feature group is gone, and so are the files. The offline store alone had saved its 15,598 rows as 10,865 small files. The other folders held Athena's answers, the training and validation files, and what the training job wrote.

Two things stay behind on purpose. The Glue table entry is only a description, it costs nothing, and quick setup's role is not allowed to delete it. The domain costs nothing either, and the next class uses one.

The space is a running machine, so stop it.

**[Please Note:]** Studio. Click **JupyterLab** at the top left. Find the space with the following name.


In [ ]:
features

On that row, click **Stop**. Wait until the status says **Stopped**.


## **11. Class summary**

A model learns from features, values worked out from raw data such as a customer's return rate. Computing them once is easy. Using them correctly in two places, for training and for live predictions, is where teams go wrong. A feature store keeps every feature row in one place, together with the time it became true.

You built that for the returns project. Every order and every known return became a moment in a customer's history, and each moment became a feature row with an `event_time`. A feature group gave those rows a home with a fixed set of typed columns, keyed by `customer_id`. Writing a row is one `put_record` call. The rows reached the offline store, files in S3, a few minutes later, and Athena read them back with SQL through the Glue table Feature Store made.

The offline store keeps every row, never overwriting. That history made the point-in-time join possible, giving each order the newest row from strictly before it. The naive join, which gives every order the customer's latest row, scored 0.833 in testing and 0.714 on the rows a live system would see. The honest training set gave a training job a model that scores 0.721, a number the shop can trust.

## **12. References and further reading**

**AWS documentation**

- [Create, store, and share features with Feature Store](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store.html). The overview, with the terms used in this class.
- [Feature Store concepts](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-concepts.html). Feature groups, record identifiers, event times, online and offline stores.
- [Offline store](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-offline.html). How rows are written to S3, the extra columns, and the Glue table.
- [Amazon Athena](https://docs.aws.amazon.com/athena/latest/ug/what-is.html). SQL over files in S3.

**Practitioner reading**

- [Feature Store, the definitive guide](https://www.hopsworks.ai/dictionary/feature-store) by Hopsworks. The problem from the side of a team that builds feature stores.
- [Point-in-time correctness](https://docs.feast.dev/getting-started/concepts/point-in-time-joins) in the Feast documentation. The same join, explained for the most used open-source feature store.

**Library reference**

- [`pandas.merge_asof`](https://pandas.pydata.org/docs/reference/api/pandas.merge_asof.html). The join used in section 8.